# Fashion Trend Intelligence - ModeTrend API Demo

Ce notebook démontre l'utilisation de l'API de segmentation vestimentaire pour l'analyse des tendances ModeTrend.

### Prérequis
1. Assurez-vous que le serveur Flask est lancé (`python web/app.py`).
2. L'API doit être accessible sur `http://127.0.0.1:5000`.

In [ ]:
import requests
import matplotlib.pyplot as plt
from PIL import Image
import io
import os

# Configuration
BASE_URL = 'http://127.0.0.1:5000'
IMG_DIR = 'imgs/IMG'
MASK_DIR = 'imgs/Mask'

## 1. Liste des images disponibles

On récupère la liste des images via l'endpoint `/api/images`.

In [ ]:
try:
    response = requests.get(f'{BASE_URL}/api/images')
    if response.status_code == 200:
        images = response.json()
        print(f"✅ Images trouvées ({len(images)})")
        print(f"Exemples : {images[:5]}...")
    else:
        print("❌ Erreur lors de la récupération des images.")
except Exception as e:
    print(f"❌ Connexion échouée : {e}. Le serveur est-il bien lancé ?")

## 2. Segmentation d'une image

On traite la première image du dataset via l'endpoint `/api/process/<image_name>`.

In [ ]:
if 'images' in locals() and images:
    image_name = images[0]
    print(f"Traitement de l'image : {image_name}...")

    process_resp = requests.get(f'{BASE_URL}/api/process/{image_name}')
    
    if process_resp.status_code == 200:
        data = process_resp.json()
        labels = data.get('labels', [])
        mask_name = data.get('mask')
        print(f"✅ Labels détectés : {', '.join(labels)}")
        
        # Visualisation
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
        
        # Image originale
        original_path = os.path.join(IMG_DIR, image_name)
        if os.path.exists(original_path):
            original = Image.open(original_path)
            ax1.imshow(original)
            ax1.set_title(f"Original : {image_name}")
            ax1.axis('off')
        
        # Masque de segmentation
        mask_path = os.path.join(MASK_DIR, mask_name)
        if os.path.exists(mask_path):
            mask = Image.open(mask_path)
            ax2.imshow(mask)
            ax2.set_title(f"Masque Segformer : {mask_name}")
            ax2.axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("❌ Erreur lors du traitement de l'image.")
else:
    print("⚠️ Aucune image à traiter.")

## 3. Analyse globale des tendances

L'endpoint `/api/analyze-all` permet d'agréger les résultats et d'identifier les pièces phares.

In [ ]:
print("Analyse globale des tendances en cours...")
try:
    trends_resp = requests.get(f'{BASE_URL}/api/analyze-all')

    if trends_resp.status_code == 200:
        trends_data = trends_resp.json()
        trends = trends_data.get('trends', {})
        
        # Tri par fréquence
        sorted_trends = dict(sorted(trends.items(), key=lambda item: item[1], reverse=True))
        
        # Graphique à barres
        plt.figure(figsize=(12, 6))
        plt.bar(sorted_trends.keys(), sorted_trends.values(), color='#6366f1')
        plt.xticks(rotation=45, ha='right')
        plt.title(f"📊 Distribution des tendances (Échantillon : {trends_data.get('total_images')} images)", fontsize=14)
        plt.xlabel("Catégorie de vêtement")
        plt.ylabel("Nombre d'occurrences")
        plt.grid(axis='y', linestyle='--', alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print("\nRésumé des tendances :")
        for cat, val in sorted_trends.items():
            print(f"- {cat}: {val} détections")
    else:
        print("❌ Erreur lors de l'analyse globale.")
except Exception as e:
    print(f"❌ Erreur : {e}")